# دليل SimulacraBench

سنستعمل المخطط النموذجي
`data/sample.json` لفهم شكل المهمة، وللسبب الأكثر تقنيةً وراء المسابقة. انظر `README.md` للاطلاع على عقد التقديم وقاعدة التسجيل واللوائح.

In [1]:
import os
import sys
from pathlib import Path

# هذا الدفتر موجود في tutorials/ بينما الأدوات في جذر المستودع،
# وكل المسارات أدناه -- config.yml وdata/sample.json و_sandbox/ -- مكتوبة
# نسبةً إلى ذلك الجذر. نحدد موقعه ونعمل انطلاقًا منه، ليعمل الدفتر بالطريقة
# نفسها سواء شُغِّل Jupyter في هذا المجلد أو في المجلد الأعلى.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "make_sandbox.py").exists()), None)
if ROOT is None:
    raise RuntimeError("شغِّل هذا الدفتر من داخل نسخة من المستودع")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

# الوحدات نفسها التي يستخدمها score.py. لا شيء هنا يعيد بناء
# برنامج التقييم: حين تسجّل نتيجة في هذا الدفتر فأنت تستدعي الشيفرة نفسها
# التي تسجّل نتيجتك.
from make_sandbox import (ROLE_COLUMN, generated_items, load_config,
                          load_schema, options_for, write_sandbox)
from score import floored, load_frames, sample_rows
from score import score as grade

SEED = 0
PHASE = 1
config = load_config("config.yml")

pd.set_option("display.width", 200, "display.max_columns", 50)

---

## ١. شكل المهمة

يحوّل `make_sandbox.py` المخطط إلى مجموعة بيانات لها الشكل نفسه تمامًا الذي
تملكه البيانات الحقيقية: الأعمدة نفسها، والخيارات نفسها، ومنطق التخطي نفسه.
أما التوزيعات الهامشية والاعتماديات فمتخيَّلة، ولذلك فإن مسارًا صُحِّح هنا
سينتقل، بينما نموذجًا ضُبِط هنا لن ينتقل.

يكتب البرنامج ملف `respondents.parquet` — كل مستجيب مع الدور `role` الذي يحدد
الغرض منه — وملف `schema.json`، وهو ما تتلقاه دالتك `predict()`. تُحدَّد
الأدوار مرة واحدة، هنا، وتُكتب على القرص. أما `score.py` فيقرأها فحسب؛ وهو لا
يقسّم شيئًا بنفسه أبدًا.

In [2]:
sample = load_schema("data/sample.json", config)
write_sandbox(sample, config, "_sandbox/sample", seed=SEED)

respondents = load_frames("_sandbox/sample", sample)
print(sample["dataset"]["description"])
print()

MEANING = {"TRAIN": "ظاهر في المرحلتين",
           "DEV": "يُسجَّل في المرحلة ١، ظاهر في المرحلة ٢",
           "TEST": "يُسجَّل في المرحلة ٢"}
counts = respondents[ROLE_COLUMN].value_counts()
print(pd.DataFrame({"المستجيبون": counts,
                    "المعنى": [MEANING[r] for r in counts.index]}).to_string())

A toy instrument, not a real survey. Ten items, few enough to print the whole schema and read it. It has one of everything the real schemas have: a frame block that is always visible, items that are scored, a gate chain two deep, and an EXCLUDE column the grader never shows anybody. The GIVEN block is deliberately the cheap half of a questionnaire -- the variables that already sit on a sampling frame, a census roster or another survey of the same households -- and the PREDICT block is the expensive half, the part that needs an enumerator and an interview. Use it to see the shape of the task; use the three real schemas to see whether a method works.

       المستجيبون                                   المعنى
role                                                      
TRAIN        8000                        ظاهر في المرحلتين
TEST         2100                     يُسجَّل في المرحلة ٢
DEV          1900  يُسجَّل في المرحلة ١، ظاهر في المرحلة ٢


### ماذا يعلن المخطط

يحمل كل بند أربعة مفاتيح: نص السؤال `question` كما يُطرح، والفئة `class`،
والقيم المسموح بها `values`، والبوابة `gate` إن كان البند لا يُطرح إلا على
بعض الأشخاص.

الفئات الثلاث هي المهمة كلها. الفئة **`GIVEN`** ظاهرة للجميع ولا تُسجَّل لها
نتيجة أبدًا. والفئة **`PREDICT`** محجوبة عن المستجيبين المحجوبين، وكل خانة
فارغة فيها تُسجَّل. أما **`EXCLUDE`** — المعرّفات ومفاتيح السجلات والنص الحر —
فلا تُسلَّم في الجدول أصلًا، لذا رشِّح حسب `class` بدل افتراض أن المخطط والجدول
يحملان الأعمدة نفسها.

في هذه الأداة يقابل التقسيمُ عمدًا النصفَ الرخيص من الاستبيان بالنصف المكلف.
فكتلة `GIVEN` هي من نوع المتغيرات الموجودة سلفًا في إطار المعاينة أو سجل
التعداد أو مسح آخر للأسر نفسها: أين يقيم الشخص، وكم حجم الأسرة، وهل لديها
هاتف. أما كتلة `PREDICT` فهي ما يتطلب باحثًا ميدانيًا ومقابلة. والجزء الثاني
قائم على هذا التمييز.

In [3]:
rows = []
for name, rec in sample["items"].items():
    gate = rec.get("gate") or {}
    rows.append({"البند": name,
                 "الفئة": rec["class"],
                 "K": len(options_for(sample, name)) if rec["values"] else 0,
                 "البوابة": gate.get("parent", "-"),
                 "يُطرح إذا": ", ".join(gate.get("observed_if", [])) or "-",
                 "الخيارات": " | ".join(map(str, rec["values"] or ["-"]))})
print(pd.DataFrame(rows).to_string(index=False))

                 البند   الفئة  K        البوابة                                             يُطرح إذا                                                الخيارات
                region   GIVEN  3              -                                                     -                                 North | Central | South
           urban_rural   GIVEN  2              -                                                     -                                           Urban | Rural
              age_band   GIVEN  4              -                                                     -                             18-29 | 30-44 | 45-59 | 60+
        household_size   GIVEN  4              -                                                     -                               1 | 2-3 | 4-5 | 6 or more
household_has_children   GIVEN  2              -                                                     -                                                Yes | No
      has_mobile_phone   GIVEN  2             

القيمة `K` هي عدد خيارات البند **متضمنًا قيمة البوابة الحارسة**: فالبند
المبوَّب له خانة أكثر مما له من إجابات، لأن «لم يُسأل قط» إجابة حقيقية بالنسبة
إليه. وهذه الخانة الإضافية هي آخر مدخل في متجه الاحتمالات لديك، ومن `K` يُحسب
المرجع المنتظم `U`.

البند `would_return` مبوَّب على `clinic_wait`، وهذا الأخير مبوَّب على
`visited_clinic`: سلسلة بعمق اثنين. فمن لم يزر عيادة قط لم يُسأل كم انتظر، ولا
سُئل إن كان سيعود. وإجابته الحقيقية عن كليهما هي `NA_GATED`.

In [4]:
chain = ["visited_clinic", "clinic_wait", "would_return"]
print(respondents[chain].value_counts().to_frame("المستجيبون").head(12).to_string())

                                                         المستجيبون
visited_clinic       clinic_wait           would_return            
No                   NA_GATED              NA_GATED            5179
Prefer not to answer NA_GATED              NA_GATED            4413
Yes                  Over 2 hours          Yes                 1405
                     Under 30 minutes      Yes                  380
                     Over 2 hours          No                   204
                                           Not sure             201
                     Under 30 minutes      Not sure             162
                                           No                    28
                     30 minutes to 2 hours Yes                   21
                                           Not sure               4
                                           No                     3


اقرأ ذلك الجدول بوصفه منطق التخطي نفسه: حيثما كانت قيمة `visited_clinic` شيئًا
غير `Yes` يكون البندان الفرعيان `NA_GATED`، دون استثناء. **إجابة البند المبوَّب
محدَّدة متى كان أصله ظاهرًا** — وهذه نقاط مجانية، وأول ما ينبغي استثماره.

### ماذا تتلقى `predict()`

يأخذ `score.py` الأدوارَ الظاهرة والمحجوبة في هذه المرحلة، ويضع المستجيبين
الظاهرين فوق المحجوبين، ويفرّغ كل خانة `PREDICT` لدى هؤلاء الأخيرين. وهذه
الخانات الفارغة هي ما تعيد له احتمالات.

القيمة `NaN` تعني شيئًا واحدًا لا غير: *هذه الخانة محجوبة، فتنبأ بها*. وهي لا
تعني أبدًا «لم يجب» — فالامتناع الحقيقي عن الإجابة خيار عادي مثل
`Prefer not to answer`، يرد في قائمة الخيارات كأي خيار آخر.

In [5]:
frame, cells, truth = sample_rows(sample, respondents, PHASE)
shown = [n for n, r in sample["items"].items() if r["class"] != "EXCLUDE"]

print("الجدول:", frame.shape, " الخانات المطلوب التنبؤ بها:", len(cells))
print()
print(pd.concat([frame[["respondent_id"] + shown].head(3),
                 frame[["respondent_id"] + shown].tail(3)]).to_string(index=False))

الجدول: (9900, 11)  الخانات المطلوب التنبؤ بها: 7600

respondent_id  region urban_rural age_band household_size household_has_children has_mobile_phone visited_clinic  clinic_wait would_return trusts_health_advice
      R000001 Central       Rural      60+            4-5                    Yes               No            Yes Over 2 hours          Yes             Somewhat
      R000002   North       Rural      60+            4-5                    Yes               No             No     NA_GATED     NA_GATED           Not at all
      R000003   South       Rural      60+            2-3                     No               No             No     NA_GATED     NA_GATED                A lot
      R009898   South       Rural      60+            2-3                    Yes               No            NaN          NaN          NaN                  NaN
      R009899   South       Rural      60+            4-5                     No              Yes            NaN          NaN          NaN        

الصفوف العليا مستجيبون ظاهرون: مكتملون، ولك أن تتعلم منهم. والصفوف السفلى
محجوبة — ترى منها كتلة `GIVEN` ولا شيء غيرها.

قيمتك المعادة هي متجه احتمالات واحد لكل خانة فارغة، وفق **الترتيب المعياري**:
الصفوف من الأعلى إلى الأسفل، وداخل الصف الواحد البنود بترتيب مفاتيح
`schema["items"]` — لا بترتيب `frame.columns` الذي قد يختلف. ويتبع كل متجه قيم
`values` الخاصة بالبند بالترتيب، مضافًا إليها خانة الحارس إن كان البند مبوَّبًا.
اقرأ الترتيب من المخطط لا من البيانات أبدًا: فالخيار الذي لم يختره أحد يشغل
خانة أيضًا.

In [6]:
print(pd.DataFrame(cells, columns=["الصف", "respondent_id", "البند"]).head(8)
      .to_string(index=False))

 الصف respondent_id                البند
 8000       R008001       visited_clinic
 8000       R008001          clinic_wait
 8000       R008001         would_return
 8000       R008001 trusts_health_advice
 8001       R008002       visited_clinic
 8001       R008002          clinic_wait
 8001       R008002         would_return
 8001       R008002 trusts_health_advice


### التسجيل

خط الأساس الجماعي: كل مستجيب محجوب يأخذ النسب الممهَّدة لكل بند، متجاهلًا كل ما
يخص الفرد. والمقياس `skill` يساوي صفرًا للتخمين المنتظم وواحدًا للكمال، وهو ما
يرتّب عليه جدول النتائج.

In [7]:
def hidden_cells(frame, items):
    '''كل خانة فارغة، بالترتيب الذي يجب أن تعيدها به predict().'''
    values = frame[items].to_numpy(dtype=object)
    ids = frame["respondent_id"].to_numpy(dtype=object)
    return [(row, ids[row], items[col])
            for row in range(values.shape[0])
            for col in range(len(items))
            if pd.isna(values[row, col])]


def crowd_for(sch, frame):
    items = generated_items(sch)
    tables = {}
    for item in items:
        counts = frame[item].value_counts()
        n = np.array([counts.get(o, 0) for o in options_for(sch, item)], float)
        tables[item] = (n + 0.5) / (n + 0.5).sum()
    return [tables[item] for _, _, item in hidden_cells(frame, items)]


vectors = floored(crowd_for(sample, frame), sample, cells, config["scoring"]["floor"])
result = grade(sample, config, vectors, truth, cells)

def table(rows):
    """Print label/value pairs, aligned however long the labels happen to be."""
    pad = max(len(label) for label, _ in rows)
    for label, value in rows:
        print("%-*s  %s" % (pad, label, value))


table([("المرجع المنتظم (نات)", "%.4f" % result["uniform_reference"]),
       ("النتيجة اللوغاريتمية", "%.4f" % result["log_score"]),
       ("skill", "%.4f" % result["skill"])])
print()
print("المقياس skill يساوي صفرًا للتخمين المنتظم وواحدًا للكمال.")

المرجع المنتظم (نات)  1.3144
النتيجة اللوغاريتمية  -0.8925
skill                 0.3210

المقياس skill يساوي صفرًا للتخمين المنتظم وواحدًا للكمال.


هذا هو العقد كله. التقديم ملف `main.py` فيه دالة `predict()` تعيد تلك
المتجهات؛ و`score.py` يشغّلها كما سيفعل برنامج التقييم، و
`tools/check_submission_zip.py` يتحقق من سلامة بنية الأرشيف الذي ترفعه.

---

## ٢. ما الذي يقدمه نموذج جيد

معيارٌ يكافئ التنبؤ بإجابات الناس يثير قلقًا بديهيًا: هل الغاية أن نكفّ عن
سؤالهم؟ نرى هنا صيغةً للجمع بين التنبؤات الخوارزمية والعينات البشرية.

نريد رقمًا واحدًا عن مجتمع سكاني: نسبة الأسر التي تثق بالنصائح الصحية الصادرة عن عيادتها المحلية. الكتلة الرخيصة — المنطقة، حضر أم ريف، حجم
الأسرة، وجود هاتف — معروفة سلفًا لكل أسرة في إطار المعاينة، من السجلات الإدارية
أو من مسح سابق. أما الكتلة المكلفة فتتطلب باحثًا ميدانيًا عند الباب، والميزانية
تكفي بضع مئات من المقابلات.

أمامك ثلاثة خيارات.

1. **المقابلات وحدها.** اسأل ٣٠٠ أسرة، وخذ النسبة، وانشر فترة ثقة. صحيح، وبدقة
   بقدر ما تسمح به ٣٠٠ مقابلة.
2. **النموذج وحده.** شغِّل نموذجًا على الكتلة الرخيصة لكل أسرة وانشر المتوسط.
   مجاني، و*خاطئ بقدر ما يخطئ النموذج* — بلا فترة ثقة وبلا سبيل إلى معرفة ذلك.
3. **كلاهما.** استعمل النموذج في كل مكان، ثم استعمل المقابلات الثلاثمائة لقياس
   خطأ النموذج وطرحه. هذا هو **الاستدلال المدعوم بالتنبؤ**، وهو ما يبنيه بقية
   هذا القسم.

الخيار الثالث هو الجدير بالاعتماد، لأنه صحيح سواء أكان النموذج جيدًا أم رديئًا،
و*أدقّ من الأول حين يكون النموذج جيدًا*.

In [8]:
# المقدار المقدَّر: نسبة من يثقون بالنصائح الصحية ولو إلى حد ما.
TARGET, POSITIVE = "trusts_health_advice", ["Somewhat", "A lot"]
GIVEN = [n for n, r in sample["items"].items() if r["class"] == "GIVEN"]

Y = respondents[TARGET].isin(POSITIVE).to_numpy(float)
design = pd.get_dummies(respondents[GIVEN].astype(str), drop_first=True)
X = np.column_stack([np.ones(len(design)), design.to_numpy(float)])

# يُدرَّب النموذج على مستجيبين من جولات سابقة -- أي الدور
# TRAIN، وهو بالضبط الكتلة الظاهرة التي يتعلم منها أي تقديم. وهو لا يرى أبدًا
# الأسر التي نوشك على مقابلتها.
past = (respondents[ROLE_COLUMN] == "TRAIN").to_numpy()
ridge = np.linalg.solve(X[past].T @ X[past] + 5 * np.eye(X.shape[1]),
                        X[past].T @ Y[past])
predicted = X @ ridge          # f(الكتلة الرخيصة)، لكل أسرة

# الإطار الذي نريد عنه رقمًا: الأسر التي لم تُستعمل في تدريب النموذج.
frame_rows = np.flatnonzero(~past)
TRUTH = Y[frame_rows].mean()   # معلوم فقط لأن البيانات متخيَّلة

table([("النموذج مدرَّب على مستجيبين سابقين، عددهم:", "%d" % past.sum()),
       ("الإطار المراد تقديره، عدد الأسر:", "%d" % len(frame_rows)),
       ("الارتباط بين التنبؤ والإجابة:", "%.2f"
        % np.corrcoef(predicted[frame_rows], Y[frame_rows])[0, 1]),
       ("النسبة الحقيقية (التي لا يراها مسح حقيقي قط):", "%.3f" % TRUTH)])

النموذج مدرَّب على مستجيبين سابقين، عددهم:     8000
الإطار المراد تقديره، عدد الأسر:               4000
الارتباط بين التنبؤ والإجابة:                  0.58
النسبة الحقيقية (التي لا يراها مسح حقيقي قط):  0.594


لنسحب الآن المقابلات الثلاثمائة ونحسب الأرقام الثلاثة.

فترة «المقابلات وحدها» هي الفترة المدرسية المعتادة. أما الفترة المدعومة بالتنبؤ
فهي متوسط النموذج على الأسر التي **لم** تقابلها، مصححًا بمتوسط خطأ النموذج على
الأسر التي قابلتها:

```
التقدير = متوسط(التنبؤ | غير مقابَلة) - [ متوسط(التنبؤ | مقابَلة) - متوسط(الإجابة | مقابَلة) ]
                ↑ النموذج مستعملًا في كل مكان        ↑ خطأ النموذج مقيسًا
```

هذا المقدار بين القوسين هو آلية الأمان كلها. يُحسب من إجابات حقيقية، ومن ثم
يكلّف مقابلات حقيقية، ويزيل تحيّز النموذج مهما يكن ذلك التحيّز.

In [9]:
def estimates(f, interviewed, rest):
    '''تقديرا المقابلات وحدها والمدعوم بالتنبؤ، ولكلٍّ خطؤه المعياري.'''
    y = Y[interviewed]
    classical = (y.mean(), y.std(ddof=1) / np.sqrt(len(y)))

    correction = f[interviewed].mean() - y.mean()
    powered = (f[rest].mean() - correction,
               np.sqrt(f[rest].var(ddof=1) / len(rest)
                       + (f[interviewed] - y).var(ddof=1) / len(interviewed)))
    return classical, powered


def band(estimate):
    point, se = estimate
    return "%.3f  [%.3f, %.3f]  العرض %.3f" % (
        point, point - 1.96 * se, point + 1.96 * se, 2 * 1.96 * se)


N_INTERVIEWS = 300
draw = np.random.default_rng(SEED).permutation(frame_rows)
interviewed, rest = draw[:N_INTERVIEWS], draw[N_INTERVIEWS:]

classical, powered = estimates(predicted, interviewed, rest)
table([("الحقيقة", "%.3f" % TRUTH),
       ("المقابلات وحدها", band(classical)),
       ("المدعوم بالتنبؤ", band(powered)),
       ("النموذج وحده (بلا مقابلات)", "%.3f  [لا فترة ثقة البتة]"
        % predicted[frame_rows].mean())])

الحقيقة                     0.594
المقابلات وحدها             0.597  [0.541, 0.652]  العرض 0.111
المدعوم بالتنبؤ             0.587  [0.541, 0.632]  العرض 0.091
النموذج وحده (بلا مقابلات)  0.590  [لا فترة ثقة البتة]


سحبة واحدة لا تثبت شيئًا — فقد تكون الفترة محظوظة. المهم هو السلوك عبر مسوح
كثيرة: هل تحتوي الفترة على القيمة الحقيقية في نحو ٩٥٪ من المرات، وكم عرضها؟
لنكرر التمرين كله ألف مرة، في كل مرة بثلاثمائة أسرة جديدة.

In [10]:
def repeat(f, n_interviews=N_INTERVIEWS, draws=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(draws):
        shuffled = rng.permutation(frame_rows)
        classical, powered = estimates(f, shuffled[:n_interviews],
                                       shuffled[n_interviews:])
        out.append(classical + powered)
    return np.array(out)          # التقدير، الخطأ المعياري، التقدير، الخطأ المعياري


def summarize(trials, label):
    rows = []
    for name, point, se in (("المقابلات وحدها", trials[:, 0], trials[:, 1]),
                            ("المدعوم بالتنبؤ", trials[:, 2], trials[:, 3])):
        rows.append({"الطريقة": name,
                     "متوسط العرض": (2 * 1.96 * se).mean(),
                     "احتواء الحقيقة": np.mean(np.abs(point - TRUTH) <= 1.96 * se)})
    out = pd.DataFrame(rows)
    print(label)
    print(out.to_string(index=False, float_format="%.3f"))
    return out


good = summarize(repeat(predicted), "نموذج يتنبأ تنبؤًا جيدًا")
narrower = 1 - good.loc[1, "متوسط العرض"] / good.loc[0, "متوسط العرض"]
print("\nالنطاق أضيق بنسبة %.0f٪، انطلاقًا من العدد نفسه من المقابلات: %d." % (100 * narrower, N_INTERVIEWS))
print("ولشراء تلك الدقة بالمقابلات وحدها لَلزم نحو %d مقابلة." % round(N_INTERVIEWS / (1 - narrower) ** 2))

نموذج يتنبأ تنبؤًا جيدًا
        الطريقة  متوسط العرض  احتواء الحقيقة
المقابلات وحدها        0.111           0.942
المدعوم بالتنبؤ        0.093           0.958

النطاق أضيق بنسبة 17٪، انطلاقًا من العدد نفسه من المقابلات: 300.
ولشراء تلك الدقة بالمقابلات وحدها لَلزم نحو 432 مقابلة.


كلتا الفترتين تحتوي على القيمة الحقيقية في نحو ٩٥٪ من المرات — وهذا ما يجعلهما
فترتَي ثقة. غير أن الفترة المدعومة بالتنبؤ **أضيق** فحسب، انطلاقًا من العمل
الميداني نفسه تمامًا. اقرأ السطر الأخير بوصفه خلاصة التمرين كله: النموذج الأفضل
لا يحذف مقابلات من الميزانية، بل يجعل كل مقابلة أثمن.

### ماذا يحدث حين يكون النموذج رديئًا

الاعتراض البديهي أن هذا لا يصحّ إلا ما دام النموذج مصيبًا، وأن الوثوق به هو
مكمن الخطر. وهذا هو الإجراء نفسه بنموذج مدرَّب على **مجتمع سكاني مختلف**.

In [11]:
from make_sandbox import make_sandbox

elsewhere = make_sandbox(sample, config, seed=99)      # مجتمع سكاني مختلف
other_design = pd.get_dummies(elsewhere[GIVEN].astype(str), drop_first=True)
other_X = np.column_stack([np.ones(len(other_design)), other_design.to_numpy(float)])
other_Y = elsewhere[TARGET].isin(POSITIVE).to_numpy(float)

wrong = np.linalg.solve(other_X.T @ other_X + 5 * np.eye(other_X.shape[1]),
                        other_X.T @ other_Y)
mispredicted = X @ wrong

table([("الارتباط بين التنبؤ والإجابة:", "%.2f"
        % np.corrcoef(mispredicted[frame_rows], Y[frame_rows])[0, 1]),
       ("النموذج وحده (بلا مقابلات)", "%.3f   مقابل حقيقة قدرها %.3f   <- بانحراف %+.3f"
        % (mispredicted[frame_rows].mean(), TRUTH,
           mispredicted[frame_rows].mean() - TRUTH))])
print()
summarize(repeat(mispredicted), "نموذج لا ينتقل")

الارتباط بين التنبؤ والإجابة:  0.06
النموذج وحده (بلا مقابلات)     0.727   مقابل حقيقة قدرها 0.594   <- بانحراف +0.133



نموذج لا ينتقل
        الطريقة  متوسط العرض  احتواء الحقيقة
المقابلات وحدها        0.111           0.942
المدعوم بالتنبؤ        0.117           0.944


,الطريقة,متوسط العرض,احتواء الحقيقة
0,المقابلات وحدها,0.111157,0.942
1,المدعوم بالتنبؤ,0.116888,0.944


لاحظ: تقدير النموذج وحده يخطئ بأكثر من العُشر، ولم يكن شيء في المخرجات لِيُنبئك
بذلك — لا فترة ثقة ولا تحذير، بل مجرد رقم يبدو في السلطة نفسها تمامًا التي يبدو
عليها الرقم الصحيح. واستبدال النموذج بالعمل الميداني يُدخل تحيّزًا.

وما زالت الفترة المدعومة بالتنبؤ تحتوي على القيمة الحقيقية في نحو ٩٥٪ من
المرات. وهي ليست أضيق من المقابلات وحدها — فالنموذج عديم الجدوى لا يشتري دقة.
وقد قاس حدّ التصحيح خطأ النموذج على المقابلات الحقيقية الثلاثمائة وطرحه، وهو
بالضبط ما وُجد من أجله.

### لماذا يحتاج هذا إلى معيار

عرض ذلك النطاق دالة مباشرة في جودة النموذج. ومن هنا تأتي الحجة لقياس جودة
التنبؤ قياسًا دقيقًا، على أدوات حقيقية، بقاعدة تسجيل سليمة: فارتفاع `skill` في
جدول النتائج يعني فترة ثقة أضيق في الميدان، أو الفترة نفسها بعدد أقل من
المقابلات.